# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You will:
- Load dataset metadata and record sets using Croissant schema
- Explore available entities by `@id` (record sets, fields, columns)
- Extract and analyze records
- Perform basic exploratory data analysis (EDA)
- Visualize results

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load Croissant metadata and records using `mlcroissant`.

- Import libraries
- Set schema URL
- Load Croissant Dataset

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For reproducibility
pd.set_option('display.max_columns', None)

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata as an object. Print name/description via attributes.
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Let's enumerate record sets and their fields using their `@id`, name, and types.

This step helps to determine which data tables are available and what kind of information you can extract.

**Note:** All entities are referenced by their `@id` field.

In [ ]:
# List all record sets with their @id, name, and field @ids
record_sets = dataset.record_sets
print(f"Total RecordSets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {getattr(rs, 'name', '(no name)')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {getattr(field, 'name', '(no name)')}, dataType: {getattr(field, 'data_type', '(not set)')}")
    print("-")

## 3. Data Extraction
Let's extract all available record sets to pandas DataFrames using their `@id`.

If you see only a single record set, just use that one (`@id`).  Display sample records and column names for further exploration.

In [ ]:
# Get all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Found record sets:")
for rid in record_set_ids:
    print(f"  - {rid}")

# Extract all available record sets into pandas DataFrames
dataframes = {}
for rid in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {rid}")
    records = list(dataset.records(record_set=rid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"  {len(df)} records, columns: {list(df.columns)}")
        display(df.head(3))
    else:
        print("  No (tabular) records found.")

## 4. Exploratory Data Analysis (EDA)
Let's select a record set with tabular data (if present), pick a numeric field, and perform:
- Filtering
- Normalization
- Grouping

We will use `@id`s for both record set and fields.


In [ ]:
# Helper: Pick first tabular record set with at least one numeric column
chosen_recordset = None
numeric_field_id = None
group_field_id = None

for rs in dataset.record_sets:
    if rs.id not in dataframes or dataframes[rs.id].empty:
        continue
    df = dataframes[rs.id]
    num_candidates = [(field.id, field.name) for field in rs.fields if getattr(field, 'data_type', None) in ['Number','Float','Integer'] and field.id in df.columns]
    group_candidates = [(field.id, field.name) for field in rs.fields if getattr(field, 'data_type', None) in ['Text', 'Boolean'] and field.id in df.columns]
    if num_candidates:
        chosen_recordset = rs.id
        # Just pick first for demo
        numeric_field_id = num_candidates[0][0]
        group_field_id = group_candidates[0][0] if group_candidates else None
        print(f"Selected record set: {chosen_recordset}, numeric field: {numeric_field_id}, group field: {group_field_id}")
        break

if not chosen_recordset:
    print("No suitable tabular record set with numeric field detected for EDA.")
else:
    df = dataframes[chosen_recordset]

    # Demonstrate filtering: Filter rows where numeric_field_id > threshold
    threshold = 0
    if numeric_field_id in df.columns:
        try:
            filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
        except Exception:
            filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records in record set {chosen_recordset} with {numeric_field_id} > {threshold}: {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
        print(f"Normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field_id if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped means of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize numeric data distribution or group means if available.

All references use `@id` fields.

In [ ]:
# Example: Histogram and (if possible) barplot for group means
if chosen_recordset and numeric_field_id and chosen_recordset in dataframes and numeric_field_id in dataframes[chosen_recordset].columns:
    df = dataframes[chosen_recordset]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].astype(float), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        # Barplot of group means
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=group_means)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated end-to-end exploration of a Croissant dataset using `mlcroissant`, referencing all entities using their `@id`s for reproducibility and clarity.

- **Metadata**: Dataset name and description loaded
- **Overview**: Listed available record sets and their field `@id`s
- **Extraction & EDA**: Loaded tabular record sets, filtered and normalized numeric fields, grouped by a categoric field
- **Visualization**: Plotted distributions and group summaries

You can now extend this notebook for deeper statistical analysis, model building, or additional visualizations!
